In [ ]:
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 13.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.2
    Uninstalling scipy-1.16.2:
      Successfully uninstalled scipy-1.16.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatib

In [ ]:
import os, re, random, numpy as np
from tqdm import tqdm
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import gensim
from gensim.models import Word2Vec

In [ ]:
# ---------------------------
# 0) Configuración y seeds
# ---------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Hiperparámetros principales
VECTOR_SIZE = 100
WINDOW      = 5
MIN_COUNT   = 5
NEGATIVE    = 10
SG          = 1       # 1 = Skip-gram, 0 = CBOW
EPOCHS_W2V  = 5
WORKERS     = max(1, os.cpu_count() or 1)

In [ ]:
# ---------------------------
# 1) Dataset IMDb (HF Datasets)
#    25k train / 25k test, binario
# ---------------------------
ds = load_dataset("imdb")  # splits: 'train', 'test', 'unsupervised'
train_ds = ds["train"]
test_ds  = ds["test"]
print(f"IMDb -> train: {len(train_ds)}, test: {len(test_ds)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


IMDb -> train: 25000, test: 25000


In [ ]:
train_ds[15000]

{'text': 'Budget limitations, time restrictions, shooting a script and then cutting it, cutting it, cutting it... This crew is a group of good, young filmmakers; thoughtful in this script - yes, allegorical - clever in zero-dollar effects when time and knowledge is all you have, relying on actors and friends and kind others for their time, devotion, locations; and getting a first feature in the can, a 1-in-1000 thing. These guys make films. Good ones. Check out their shorts collection "Heartland Horrors" and see the development. And I can vouch, working with them is about the most fun thing you\'ll do in the business. I\'m stymied by harsh, insulting criticism for this film, wondering if one reviewer even heard one word of dialogue, pondered one thought or concept, or if all that was desired of this work was the visual gore of bashing and slashing to satisfy some mindless view of what horror should mean to an audience. Let "The Empty Acre" bring itself to you. Don\'t preconceive what y

In [ ]:
# ---------------------------
# 2) Tokenizador simple
# ---------------------------
_word_re = re.compile(r"\b\w+\b", re.UNICODE)

def tokenize(text: str):
    return _word_re.findall(text.lower())

# Pre-tokenizamos para entrenar W2V (solo textos de train)
sentences = [tokenize(ex["text"]) for ex in tqdm(train_ds, desc="Tokenizando train")]
print("Ejemplo de oración tokenizada:", sentences[15000][:20])

Tokenizando train: 100%|██████████| 25000/25000 [00:03<00:00, 8213.53it/s]


Ejemplo de oración tokenizada: ['budget', 'limitations', 'time', 'restrictions', 'shooting', 'a', 'script', 'and', 'then', 'cutting', 'it', 'cutting', 'it', 'cutting', 'it', 'this', 'crew', 'is', 'a', 'group']


In [ ]:
# ---------------------------
# 3) Entrenar Word2Vec (gensim)
# ---------------------------
w2v = Word2Vec(
    sentences=sentences,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=WORKERS,
    sg=SG,
    negative=NEGATIVE,
    epochs=EPOCHS_W2V,
)
print("Vocab size (w2v):", len(w2v.wv.key_to_index))

Vocab size (w2v): 29114


In [ ]:
dist_1 = w2v.wv.similarity("good", "great")
dist_2 = w2v.wv.similarity("good", "accident")
print(f"Distancia coseno entre 'good' y 'great': {dist_1:.4f}")
print(f"Distancia coseno entre 'good' y 'accident': {dist_2:.4f}")

similar_words = w2v.wv.most_similar(positive=["great"], topn=3)
print(f"Palabras similares a 'great': {similar_words}")

word = w2v.wv.most_similar(positive=["james", "bruce"], negative=["cameron"], topn=1)[0][0]
print(f"Analogía 'james' - 'cameron' + 'bruce' → palabra más cercana: {word}")

Distancia coseno entre 'good' y 'great': 0.8113
Distancia coseno entre 'good' y 'horror': 0.1002
Palabras similares a 'great': [('wonderful', 0.8505077958106995), ('fantastic', 0.8264436721801758), ('good', 0.8113303184509277)]
Analogía 'james' - 'cameron' + 'bruce' → palabra más cercana: willis


In [ ]:
# ---------------------------
# 4) Función: vector de documento
#    (promedio de embeddings de sus palabras)
# ---------------------------
def doc_vector(tokens):
    # filtrar palabras fuera de vocab
    vecs = [w2v.wv[w] for w in tokens if w in w2v.wv.key_to_index]
    if not vecs:
        return np.zeros(w2v.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

# Construimos matrices X/y
def build_xy(split, desc):
    X, y = [], []
    for ex in tqdm(split, desc=desc):
        tokens = tokenize(ex["text"])
        X.append(doc_vector(tokens))
        y.append(int(ex["label"]))  # 0 = neg, 1 = pos
    return np.vstack(X), np.array(y, dtype=np.int64)

X_train, y_train = build_xy(train_ds, "Vectorizando train")
X_test,  y_test  = build_xy(test_ds,  "Vectorizando test")

print("Shapes -> X_train:", X_train.shape, " | X_test:", X_test.shape)

Vectorizando test: 100%|██████████| 25000/25000 [00:16<00:00, 1551.46it/s]


Shapes -> X_train: (25000, 100)  | X_test: (25000, 100)


In [ ]:
# ---------------------------
# 5) Clasificador (LogReg)
# ---------------------------
clf = LogisticRegression(max_iter=1000, n_jobs=-1)
clf.fit(X_train, y_train)

# Eval
pred_train = clf.predict(X_train)
pred_test  = clf.predict(X_test)

acc_train = accuracy_score(y_train, pred_train) * 100
acc_test  = accuracy_score(y_test,  pred_test)  * 100

print(f"Accuracy -> train: {acc_train:.2f}% | test: {acc_test:.2f}%\n")

Accuracy -> train: 83.61% | test: 83.00%



In [ ]:
# ---------------------------
# 6) Inferencia de ejemplo
# ---------------------------
ejemplos = [
    "I absolutely loved this movie, the acting and the story were fantastic.",
    "Boring and predictable. I wouldn't recommend it."
]
for txt in ejemplos:
    v = doc_vector(tokenize(txt)).reshape(1, -1)
    pred = clf.predict(v)[0]
    print(f"Texto: {txt[:60]}...  -> Predicción: {'pos' if pred==1 else 'neg'}")

Texto: I absolutely loved this movie, the acting and the story were...  -> Predicción: pos
Texto: Boring and predictable. I wouldn't recommend it....  -> Predicción: neg


Ejercicios:
1) Comparar Skip-gram (sg=1) vs CBOW (sg=0), y probar distintos valores para window, vector_size, negative, epochs.
2) Hacer un grafico de los embeddings de palabras usando MDS o T-sne de sklearn. ¿Se observa algun patrón?
3) Entrenar 2 word2vec: uno con las reviews positivas y otro con las negativas. Comparar las distancias entre palabras.
